# 04 — Baseline LSTM Training

This notebook runs `src/train_LSTM_baseline.py`, which executes the full Phase 3a pipeline:

1. Load the train / val / test splits from `data/processed/`.
2. Select input feature columns (configurable from the CLI).
3. Log-transform the target `realized_vol_21d` for training.
4. Fit a `StandardScaler` on the train features.
5. Build sliding-window `DataLoader`s.
6. Run an **Optuna** study (search space in `config.LSTM_SEARCH_SPACE`) — each trial is scored by validation MSE back in the *original* (inverse-log) volatility scale.
7. Retrain the model with the best hyperparameters and evaluate it on the test set (MSE / RMSE / MAE, original scale).
8. Save artifacts to `models/lstm_baseline.pt` and `models/lstm_baseline_scaler.joblib`.

The defaults match the feature list requested for the baseline: `Open, High, Low, Close, Volume, log_return, abs_return, oc_return, intraday_range, log_volume`. Pass `--features ...` to override.

In [1]:
import json
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
SCRIPT = REPO_ROOT / "src" / "train_LSTM_baseline.py"
assert SCRIPT.exists(), f"Missing: {SCRIPT}"
print("Repo root:", REPO_ROOT)
print("Script   :", SCRIPT)

Repo root: /Users/changjiangliu/Desktop/StockVolatilitySight
Script   : /Users/changjiangliu/Desktop/StockVolatilitySight/src/train_LSTM_baseline.py


## Run the training pipeline

Tweak `ARGS` below to change the trial count, epochs, or feature list. The defaults come from `config.py` (`LSTM_N_TRIALS`, `LSTM_TUNE_EPOCHS`, `LSTM_FINAL_EPOCHS`, `LSTM_BASELINE_FEATURES`).

In [2]:
ARGS = [
    # "--features", "Close", "Volume", "log_return", "abs_return", "log_volume",
    # "--n-trials", "20",
    # "--tune-epochs", "30",
    # "--final-epochs", "80",
]

cmd = [sys.executable, str(SCRIPT), *ARGS]
print("Running:", " ".join(cmd))
print("-" * 80)

proc = subprocess.run(cmd, cwd=str(REPO_ROOT), capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"train_LSTM_baseline.py exited with code {proc.returncode}")

Running: /Users/changjiangliu/Desktop/StockVolatilitySight/venv/bin/python /Users/changjiangliu/Desktop/StockVolatilitySight/src/train_LSTM_baseline.py
--------------------------------------------------------------------------------


KeyboardInterrupt: 

## Parse the results

The script prints a JSON block under `=== Baseline LSTM results ===`. We extract it here for easy display in downstream analysis.

In [ ]:
marker = "=== Baseline LSTM results ==="
idx = proc.stdout.find(marker)
assert idx != -1, "Results marker not found in script output."
json_blob = proc.stdout[idx + len(marker):].strip()
results = json.loads(json_blob)

print("Best hyperparameters:")
for k, v in results["best_params"].items():
    print(f"  {k:>12}: {v}")

print(f"\nValidation MSE (raw scale, Optuna best) : {results['best_val_mse_raw']:.8f}")
print(f"Validation MSE (raw scale, final retrain): {results['retrained_val_mse_raw']:.8f}")

print("\nTest metrics (raw scale):")
for k, v in results["test_metrics"].items():
    print(f"  {k:>6}: {v}")

print(f"\nFeatures used ({len(results['features'])}): {results['features']}")
print(f"Target        : {results['target']}")
print(f"N trials      : {results['n_trials']}")

## Saved artifacts

- `models/lstm_baseline.pt` — model `state_dict`, selected hyperparameters, feature list.
- `models/lstm_baseline_scaler.joblib` — the `StandardScaler` fit on the train features (needed to reproduce predictions).

Reload example:

```python
import torch, joblib
from src.lstm_model import LSTMRegressor
ckpt = torch.load('../models/lstm_baseline.pt', weights_only=False)
model = LSTMRegressor(input_size=ckpt['n_features'], **{k: ckpt['hyperparameters'][k] for k in ('hidden_size','n_layers','dropout')})
model.load_state_dict(ckpt['state_dict'])
scaler = joblib.load('../models/lstm_baseline_scaler.joblib')
```